# 🚜 FieldOps — Análise de Telemetria GPS por Área Agrícola

Este notebook do **Google Colab** identifica em qual área de um **KML** (fazendas,
talhões, setores ou blocos) cada ponto GPS de telemetria ocorreu e calcula
**quanto tempo cada equipamento trabalhou dentro de cada área**.

**Entradas:**
1. Um arquivo **KML** com polígonos das áreas agrícolas.
2. Um arquivo **ZIP** contendo um ou mais CSVs de telemetria com as colunas:
   `ceqid, nickname, vin, name, numeric_value, text_value, uom, event_timestamp, lat, lon`.

**Saídas:**
- `resultado_pontos.xlsx` — todos os pontos com a área associada e o tempo calculado.
- `resumo_por_area.xlsx` — horas trabalhadas, pontos e equipamentos por área.
- `resumo_por_equipamento.xlsx` — horas e pontos por equipamento e área.
- `mapa.html` — mapa interativo (Folium) com polígonos e pontos.

> **Como usar:** execute as células **na ordem**, de cima para baixo
> (`Runtime → Run all` ou `Ctrl/Cmd + Enter` célula a célula).

O código usa **GeoPandas Spatial Join**, **índice espacial** e **operações
vetorizadas**, suportando centenas de milhares de pontos sem loops linha a linha.


## ⚙️ Etapa 1 — Instalação automática das dependências

Instala todas as bibliotecas necessárias. Compatível com o ambiente do Google
Colab. Pode levar 1–2 minutos na primeira execução.

> Se aparecer um aviso pedindo para **reiniciar o ambiente** ("Restart runtime"),
> reinicie e execute novamente a partir daqui.


In [ ]:
# Instalacao silenciosa das dependencias no Google Colab.
# pyogrio + fiona dao suporte de leitura/escrita vetorial ao GeoPandas.
import sys, subprocess

PACOTES = [
    "pandas",
    "geopandas",
    "shapely",
    "fiona",
    "pyogrio",
    "fastkml",
    "folium",
    "openpyxl",
    "tqdm",
    "lxml",          # backend usado pelo fastkml para parsear o KML
]

print("Instalando dependencias (isso pode levar alguns minutos)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *PACOTES],
    check=True,
)
print("Dependencias instaladas com sucesso.")


### Importações e configurações globais

Importa as bibliotecas e define algumas constantes usadas no restante do notebook.


In [ ]:
import os
import re
import glob
import zipfile
import warnings
from io import BytesIO

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely.validation import make_valid
from tqdm.auto import tqdm

import folium

warnings.filterwarnings("ignore")
tqdm.pandas()

# ----------------------------- Parametros -----------------------------------
EPSG_GEO = 4326                 # WGS84 (lat/lon)
MAX_GAP_SECONDS = 10 * 60       # Intervalo maximo valido entre pontos = 10 minutos
COLUNAS_OBRIGATORIAS = ["ceqid", "lat", "lon", "event_timestamp"]

# ============================================================================
# CONFIGURACAO DOS ESTADOS DE OPERACAO (a partir da coluna 'name')
# ----------------------------------------------------------------------------
# A telemetria vem em formato "longo": cada linha e UMA leitura de um sinal,
# identificado pela coluna 'name', com o valor em 'numeric_value' ou 'text_value'.
# Aqui definimos quais sinais representam cada estado. A correspondencia ignora
# acentos e maiusculas/minusculas (ex.: "Status da orientacao automatica").
#
# Se os nomes dos seus sinais forem diferentes, ajuste as palavras-chave abaixo.
# A Etapa 3.5 imprime os nomes de sinais encontrados para facilitar o ajuste.
# ============================================================================

# --- TEMPO EFETIVO = carga de motor >= 35% E status do elevador = "forward" ---
LIMIAR_CARGA_MOTOR = 35.0                       # % minima de carga do motor
SINAL_CARGA_MOTOR = ["carga", "motor"]          # 'name' deve conter TODAS estas palavras
SINAL_STATUS_ELEVADOR = ["elevador"]            # 'name' deve conter estas palavras
VALOR_ELEVADOR_EFETIVO = ["forward"]            # valor (text_value) que indica efetivo

# --- PILOTO LIGADO = "Status da orientacao automatica" = on ------------------
SINAL_PILOTO = ["orientacao", "automatica"]     # 'name' = Status da orientacao automatica
VALORES_LIGADO = ["on", "ligado", "ativo", "ativado", "engaged",
                  "true", "sim", "yes", "1", "habilitado"]

# Pasta de trabalho onde os arquivos serao salvos
WORKDIR = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(WORKDIR)


# --------------------------- Funcoes auxiliares -----------------------------
def remover_acentos(texto):
    """Normaliza string: minuscula, sem acentos e sem espacos extras."""
    import unicodedata
    if texto is None:
        return ""
    s = str(texto)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s.lower().strip()


def corrigir_mojibake(texto):
    """Tenta reparar acentuacao quebrada (mojibake) do tipo 'orientaÃ§Ã£o' -> 'orientação'.

    Ocorre quando um texto UTF-8 foi lido como latin-1/cp1252. So aplica a
    correcao quando ela claramente melhora o texto (remove os caracteres 'Ã/Â').
    """
    if not isinstance(texto, str):
        return texto
    if "Ã" not in texto and "Â" not in texto:
        return texto
    try:
        reparado = texto.encode("latin-1").decode("utf-8")
        return reparado
    except (UnicodeEncodeError, UnicodeDecodeError):
        return texto


def contem_todas(serie_norm, palavras):
    """Mascara booleana: True onde a serie normalizada contem TODAS as palavras."""
    mask = pd.Series(True, index=serie_norm.index)
    for p in palavras:
        mask &= serie_norm.str.contains(remover_acentos(p), na=False, regex=False)
    return mask


print("Versoes:")
print("  pandas    :", pd.__version__)
print("  geopandas :", gpd.__version__)
print("Diretorio de trabalho:", WORKDIR)


## 📤 Etapa 2 — Upload dos arquivos

Faça o upload manual dos dois arquivos pelo navegador:

- **KML** com os polígonos das áreas (`.kml`)
- **ZIP** com os CSVs de telemetria (`.zip`)

Você pode selecionar os dois de uma vez ou rodar a célula duas vezes.


In [ ]:
# Upload manual via navegador. Funciona apenas no Google Colab.
try:
    from google.colab import files
    EM_COLAB = True
except ImportError:
    EM_COLAB = False
    print("Aviso: nao parece estar no Google Colab.")
    print("Coloque 'arquivo.kml' e 'arquivo.zip' no diretorio de trabalho manualmente.")

if EM_COLAB:
    print("Selecione o arquivo .KML e o arquivo .ZIP (pode selecionar os dois juntos):")
    enviados = files.upload()
    for nome in enviados:
        print(f"  recebido: {nome} ({len(enviados[nome])/1024:.1f} KB)")

# Localiza automaticamente os arquivos enviados no diretorio de trabalho
def _encontrar(extensao):
    arquivos = sorted(glob.glob(os.path.join(WORKDIR, f"*{extensao}")))
    return arquivos

kmls = _encontrar(".kml")
zips = _encontrar(".zip")

if not kmls:
    raise FileNotFoundError("Nenhum arquivo .kml encontrado. Faca o upload do KML.")
if not zips:
    raise FileNotFoundError("Nenhum arquivo .zip encontrado. Faca o upload do ZIP.")

KML_PATH = kmls[0]
ZIP_PATH = zips[0]
print("\nArquivos selecionados:")
print("  KML:", KML_PATH)
print("  ZIP:", ZIP_PATH)
if len(kmls) > 1:
    print("  (varios KMLs encontrados; usando o primeiro)")
if len(zips) > 1:
    print("  (varios ZIPs encontrados; usando o primeiro)")


## 📖 Etapa 3 — Leitura dos dados

### 3.1 — Leitura do KML → GeoDataFrame

Lê o KML, extrai **todos os polígonos** e **todos os atributos** disponíveis
(nome, descrição, e quaisquer campos de *ExtendedData*). O notebook tenta
primeiro o leitor nativo do GeoPandas (`pyogrio`/`fiona`) e, se falhar, usa um
parser manual baseado em `fastkml`/`lxml`, garantindo robustez contra KMLs
com estruturas variadas.


In [ ]:
def ler_kml_geopandas(caminho):
    """Tenta ler o KML usando os drivers nativos do GeoPandas (pyogrio/fiona).

    O driver KML do GDAL nem sempre ativa por padrao; habilitamos o suporte.
    Retorna um GeoDataFrame ou levanta excecao.
    """
    # Habilita o driver KML no fiona/GDAL (necessario em alguns ambientes)
    try:
        import fiona
        fiona.drvsupport.supported_drivers["KML"] = "rw"
        fiona.drvsupport.supported_drivers["LIBKML"] = "rw"
    except Exception:
        pass

    ultima_excecao = None
    # pyogrio costuma ler todas as camadas/pastas do KML de uma vez
    for engine in ("pyogrio", "fiona"):
        try:
            gdf = gpd.read_file(caminho, engine=engine)
            if len(gdf) > 0:
                return gdf
        except Exception as e:  # tenta o proximo engine
            ultima_excecao = e
    if ultima_excecao:
        raise ultima_excecao
    raise ValueError("KML lido, porem sem feicoes.")


def ler_kml_fastkml(caminho):
    """Parser manual de KML usando lxml diretamente.

    Percorre todos os <Placemark>, lendo <name>, <description>, os campos de
    <ExtendedData> e a geometria (Polygon / MultiGeometry). Mais tolerante a
    KMLs nao padronizados.
    """
    from lxml import etree
    from shapely.geometry import Polygon, MultiPolygon

    with open(caminho, "rb") as f:
        conteudo = f.read()
    # Remove declaracao de encoding duplicada e faz parse tolerante
    parser = etree.XMLParser(recover=True, huge_tree=True)
    root = etree.fromstring(conteudo, parser=parser)

    def localname(tag):
        return etree.QName(tag).localname if tag is not None else ""

    def parse_coords(texto):
        pts = []
        for token in texto.replace("\n", " ").split():
            partes = token.split(",")
            if len(partes) >= 2:
                try:
                    lon, lat = float(partes[0]), float(partes[1])
                    pts.append((lon, lat))
                except ValueError:
                    continue
        return pts

    registros = []
    geometrias = []
    for pm in root.iter():
        if localname(pm.tag) != "Placemark":
            continue
        atributos = {}
        # nome e descricao
        for child in pm:
            ln = localname(child.tag)
            if ln == "name":
                atributos["name"] = (child.text or "").strip()
            elif ln == "description":
                atributos["description"] = (child.text or "").strip()

        # ExtendedData -> Data/SimpleData
        for data in pm.iter():
            ln = localname(data.tag)
            if ln in ("Data", "SimpleData"):
                nome_attr = data.get("name")
                if nome_attr:
                    valor = ""
                    if ln == "SimpleData":
                        valor = (data.text or "").strip()
                    else:  # <Data><value>...</value></Data>
                        for v in data:
                            if localname(v.tag) == "value":
                                valor = (v.text or "").strip()
                    atributos[nome_attr] = valor

        # Geometria: coleta todos os aneis de Polygon
        poligonos = []
        for poly in pm.iter():
            if localname(poly.tag) != "Polygon":
                continue
            exterior = None
            interiores = []
            for ring in poly.iter():
                if localname(ring.tag) == "outerBoundaryIs":
                    for c in ring.iter():
                        if localname(c.tag) == "coordinates":
                            exterior = parse_coords(c.text or "")
                elif localname(ring.tag) == "innerBoundaryIs":
                    for c in ring.iter():
                        if localname(c.tag) == "coordinates":
                            interiores.append(parse_coords(c.text or ""))
            if exterior and len(exterior) >= 3:
                poligonos.append(Polygon(exterior, interiores))

        if not poligonos:
            continue
        geom = poligonos[0] if len(poligonos) == 1 else MultiPolygon(poligonos)
        registros.append(atributos)
        geometrias.append(geom)

    if not geometrias:
        raise ValueError("Nenhum poligono encontrado no KML (parser fastkml/lxml).")

    gdf = gpd.GeoDataFrame(registros, geometry=geometrias, crs=f"EPSG:{EPSG_GEO}")
    return gdf


# ----------------------------- Execucao -------------------------------------
try:
    try:
        areas_gdf = ler_kml_geopandas(KML_PATH)
        print("KML lido com o driver nativo do GeoPandas.")
    except Exception as e1:
        print("Driver nativo falhou (", repr(e1), "). Tentando parser fastkml/lxml...")
        areas_gdf = ler_kml_fastkml(KML_PATH)
        print("KML lido com o parser fastkml/lxml.")
except Exception as e:
    raise RuntimeError(
        "Nao foi possivel ler o KML. Verifique se o arquivo e valido. Erro: " + repr(e)
    )

# Mantem apenas geometrias de area (Polygon / MultiPolygon)
areas_gdf = areas_gdf[areas_gdf.geometry.notna()].copy()
areas_gdf = areas_gdf[areas_gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
if areas_gdf.empty:
    raise ValueError("O KML nao contem poligonos (apenas pontos/linhas?).")

# Garante CRS WGS84
if areas_gdf.crs is None:
    areas_gdf = areas_gdf.set_crs(epsg=EPSG_GEO)
else:
    areas_gdf = areas_gdf.to_crs(epsg=EPSG_GEO)

# Corrige geometrias invalidas (auto-interseccao etc.) de forma vetorizada
invalidas = ~areas_gdf.geometry.is_valid
if invalidas.any():
    print(f"Corrigindo {int(invalidas.sum())} geometria(s) invalida(s) no KML...")
    areas_gdf.loc[invalidas, "geometry"] = areas_gdf.loc[invalidas, "geometry"].apply(make_valid)

areas_gdf = areas_gdf.reset_index(drop=True)
print(f"\nTotal de areas (poligonos) lidas: {len(areas_gdf)}")
print("Colunas/atributos disponiveis no KML:", list(areas_gdf.columns))
areas_gdf.head()


### 3.2 — Padronização do nome e do ID da área

Define duas colunas estáveis — `area_nome` e `area_id` — usadas no restante do
notebook, escolhendo automaticamente a melhor coluna disponível para "nome" da
área. Todos os atributos originais do KML são preservados.


In [ ]:
# Escolhe a melhor coluna para representar o NOME da area
CANDIDATAS_NOME = ["name", "Name", "nome", "NOME", "talhao", "Talhao",
                   "TALHAO", "setor", "bloco", "area", "AREA", "label", "title"]

col_nome = None
for c in CANDIDATAS_NOME:
    if c in areas_gdf.columns:
        # so usa se tiver pelo menos um valor nao vazio
        if areas_gdf[c].astype(str).str.strip().replace("nan", "").any():
            col_nome = c
            break

if col_nome is None:
    areas_gdf["area_nome"] = ["Area_" + str(i + 1) for i in range(len(areas_gdf))]
    print("Nenhuma coluna de nome encontrada; gerando nomes sequenciais (Area_1, Area_2, ...).")
else:
    areas_gdf["area_nome"] = (
        areas_gdf[col_nome].astype(str).str.strip().replace({"nan": "", "None": ""})
    )
    # preenche nomes vazios com sequencial
    vazios = areas_gdf["area_nome"] == ""
    areas_gdf.loc[vazios, "area_nome"] = [
        "Area_" + str(i + 1) for i in areas_gdf.index[vazios]
    ]
    print(f"Coluna de nome da area: '{col_nome}'")

# ID estavel da area (indice inteiro). Mantemos tambem como coluna.
areas_gdf["area_id"] = areas_gdf.index.astype(int)

# Lista de atributos extras do KML (tudo que nao e geometria/derivado)
ATRIBUTOS_KML = [
    c for c in areas_gdf.columns
    if c not in ("geometry", "area_id", "area_nome")
]
print("Atributos extras do KML que serao anexados aos pontos:", ATRIBUTOS_KML)
areas_gdf[["area_id", "area_nome"] + ATRIBUTOS_KML].head()


### 3.3 — Leitura do ZIP de telemetria → DataFrame consolidado

Abre o ZIP automaticamente, lê **todos os CSVs** (mesmo em subpastas),
consolida em um único DataFrame e valida as colunas obrigatórias.


In [ ]:
def ler_zip_csvs(caminho_zip):
    """Le todos os CSVs de dentro de um ZIP e consolida em um unico DataFrame.

    Tenta diferentes separadores e encodings para robustez.
    """
    if not zipfile.is_zipfile(caminho_zip):
        raise ValueError(f"'{caminho_zip}' nao e um arquivo ZIP valido.")

    frames = []
    with zipfile.ZipFile(caminho_zip) as zf:
        nomes_csv = [
            n for n in zf.namelist()
            if n.lower().endswith(".csv") and not n.startswith("__MACOSX")
        ]
        if not nomes_csv:
            raise ValueError("O ZIP nao contem nenhum arquivo .csv.")

        print(f"{len(nomes_csv)} CSV(s) encontrado(s) no ZIP.")
        for nome in tqdm(nomes_csv, desc="Lendo CSVs"):
            dados = zf.read(nome)
            df_csv = None
            ultima_exc = None
            # Tenta combinacoes de separador/encoding mais comuns
            for sep in (",", ";", "\t"):
                for enc in ("utf-8", "latin-1"):
                    try:
                        tmp = pd.read_csv(
                            BytesIO(dados), sep=sep, encoding=enc,
                            low_memory=False,
                        )
                        # heuristica: precisa ter mais de 1 coluna
                        if tmp.shape[1] > 1:
                            df_csv = tmp
                            break
                    except Exception as e:
                        ultima_exc = e
                if df_csv is not None:
                    break
            if df_csv is None:
                print(f"  aviso: nao foi possivel ler '{nome}' ({ultima_exc}); ignorado.")
                continue
            df_csv["__arquivo_origem"] = os.path.basename(nome)
            frames.append(df_csv)

    if not frames:
        raise ValueError("Nenhum CSV pode ser lido do ZIP.")

    df = pd.concat(frames, ignore_index=True)
    # Normaliza nomes de colunas (remove espacos e padroniza minusculas)
    df.columns = [str(c).strip() for c in df.columns]
    return df


try:
    df = ler_zip_csvs(ZIP_PATH)
except Exception as e:
    raise RuntimeError("Falha ao abrir/ler o ZIP: " + repr(e))

print(f"\nTotal de linhas consolidadas: {len(df):,}".replace(",", "."))
print("Colunas encontradas:", list(df.columns))

# Validacao das colunas obrigatorias
faltando = [c for c in COLUNAS_OBRIGATORIAS if c not in df.columns]
if faltando:
    raise ValueError(
        "Os CSVs nao possuem as colunas obrigatorias: " + ", ".join(faltando) +
        ".\nColunas presentes: " + ", ".join(df.columns)
    )

# Garante a existencia das colunas opcionais esperadas (preenche vazio se faltar)
for c in ["nickname", "vin", "name", "numeric_value", "text_value", "uom"]:
    if c not in df.columns:
        df[c] = np.nan

df.head()


### 3.4 — Validação e limpeza dos registros

Converte `lat`, `lon` e `event_timestamp` para os tipos corretos e **remove
registros inválidos**: coordenadas ausentes/fora de faixa, timestamps que não
podem ser interpretados, e pontos em (0, 0).


In [ ]:
n_inicial = len(df)

# Converte lat/lon para numerico (valores nao numericos viram NaN)
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# Converte o timestamp. utc=True garante consistencia de fuso.
df["event_timestamp"] = pd.to_datetime(
    df["event_timestamp"], errors="coerce", utc=True
)

# Mascara de registros validos (operacao vetorizada)
valido = (
    df["lat"].notna() & df["lon"].notna() &
    df["event_timestamp"].notna() &
    df["lat"].between(-90, 90) & df["lon"].between(-180, 180) &
    ~((df["lat"] == 0) & (df["lon"] == 0))            # descarta (0,0)
)

removidos = int((~valido).sum())
df = df[valido].copy().reset_index(drop=True)

# Normaliza ceqid como string (identificador do equipamento)
df["ceqid"] = df["ceqid"].astype(str).str.strip()

# Corrige acentuacao quebrada (mojibake) nas colunas de texto
# Ex.: "Status da orientaÃ§Ã£o automÃ¡tica" -> "Status da orientação automática"
# Usa mapa por valores unicos (rapido mesmo com milhoes de linhas).
for col in ["name", "text_value", "nickname"]:
    if col in df.columns:
        unicos = df[col].dropna().unique()
        mapa_fix = {v: corrigir_mojibake(v) for v in unicos}
        df[col] = df[col].map(mapa_fix).where(df[col].notna(), df[col])

print(f"Registros iniciais : {n_inicial:,}".replace(",", "."))
print(f"Registros removidos: {removidos:,}".replace(",", "."))
print(f"Registros validos  : {len(df):,}".replace(",", "."))
if df.empty:
    raise ValueError("Nenhum registro valido restante apos a limpeza.")
df.head()


### 3.5 — Reconstrução dos estados de operação (sinais da coluna `name`)

A telemetria está em **formato longo**: cada linha é uma leitura de **um** sinal
(coluna `name`), com o valor em `numeric_value` ou `text_value`. Para calcular os
tempos por estado, reconstruímos uma tabela com **uma linha por amostra**
(`ceqid` + `event_timestamp` + posição) e, nela, o valor de cada sinal de interesse:

- **Carga do motor** (`numeric_value`, %) — sinal cujo `name` contém *carga* e *motor*.
- **Status do elevador** (`text_value`) — sinal cujo `name` contém *elevador*.
- **Status da orientação automática / piloto** (`text_value`) — `name` contém *orientação automática*.

Com base nesses sinais marcamos dois estados:

| Estado | Regra |
|---|---|
| **Tempo efetivo** | carga do motor **≥ 35%** **E** status do elevador = `forward` |
| **Piloto ligado** | Status da orientação automática = `on` (ligado) |

Como esses sinais costumam ser registrados **apenas quando mudam**, aplicamos
*forward-fill* (último valor conhecido) por equipamento ao longo do tempo, para
que o estado persista até a próxima mudança.


In [ ]:
# Versao normalizada (sem acento/maiuscula) da coluna 'name' para casar sinais.
# Mapa por valores unicos -> rapido mesmo com milhoes de linhas.
mapa_norm = {v: remover_acentos(v) for v in df["name"].dropna().unique()}
nome_norm = df["name"].map(mapa_norm).fillna("")

# Identifica as linhas de cada sinal de interesse (mascaras vetorizadas)
mask_carga = contem_todas(nome_norm, SINAL_CARGA_MOTOR)
mask_elev = contem_todas(nome_norm, SINAL_STATUS_ELEVADOR)
mask_piloto = contem_todas(nome_norm, SINAL_PILOTO)

# Diagnostico: mostra os nomes de sinais encontrados para cada estado
def _amostra_nomes(mask):
    if mask.any():
        return sorted(df.loc[mask, "name"].dropna().astype(str).unique())[:5]
    return []

print("Sinais identificados na coluna 'name':")
print("  Carga do motor      :", _amostra_nomes(mask_carga) or "NENHUM (ajuste SINAL_CARGA_MOTOR)")
print("  Status do elevador  :", _amostra_nomes(mask_elev) or "NENHUM (ajuste SINAL_STATUS_ELEVADOR)")
print("  Orientacao automatica:", _amostra_nomes(mask_piloto) or "NENHUM (ajuste SINAL_PILOTO)")
if not (mask_carga.any() or mask_elev.any() or mask_piloto.any()):
    print("\n  >> Top 20 nomes de sinais presentes (para ajustar a configuracao):")
    for nm in df["name"].dropna().astype(str).value_counts().head(20).index:
        print("     -", nm)

# ------------------------------------------------------------------
# Monta a tabela de AMOSTRAS: uma linha por (ceqid, event_timestamp)
# ------------------------------------------------------------------
chave = ["ceqid", "event_timestamp"]

# Valores representativos por amostra (posicao e identificadores)
samples = (
    df.sort_values(chave)
    .groupby(chave, as_index=False)
    .agg(
        lat=("lat", "first"),
        lon=("lon", "first"),
        nickname=("nickname", "first"),
        vin=("vin", "first"),
    )
)

def _valor_sinal(mask, coluna):
    """Valor do sinal (coluna) por amostra (ceqid,event_timestamp). Pega o 1o por chave."""
    if not mask.any():
        return pd.Series(index=pd.MultiIndex.from_arrays([[], []], names=chave), dtype=object)
    sub = df.loc[mask, chave + [coluna]].dropna(subset=[coluna])
    if sub.empty:
        return pd.Series(index=pd.MultiIndex.from_arrays([[], []], names=chave), dtype=object)
    return sub.groupby(chave)[coluna].first()

# Extrai o valor de cada sinal por amostra e junta na tabela de amostras
carga_serie = pd.to_numeric(_valor_sinal(mask_carga, "numeric_value"), errors="coerce")
elev_serie = _valor_sinal(mask_elev, "text_value")
piloto_serie = _valor_sinal(mask_piloto, "text_value")

samples = samples.set_index(chave)
samples["carga_motor"] = carga_serie
samples["status_elevador"] = elev_serie
samples["status_piloto"] = piloto_serie
samples = samples.reset_index()

# Ordena por equipamento e tempo e aplica forward-fill por equipamento
# (o estado persiste ate a proxima leitura que o altere)
samples = samples.sort_values(chave).reset_index(drop=True)
for col in ["carga_motor", "status_elevador", "status_piloto"]:
    samples[col] = samples.groupby("ceqid")[col].ffill()

# ------------------------------------------------------------------
# Marca os estados de operacao (flags booleanas vetorizadas)
# ------------------------------------------------------------------
_map_elev = {v: remover_acentos(v) for v in samples["status_elevador"].dropna().unique()}
_map_pil = {v: remover_acentos(v) for v in samples["status_piloto"].dropna().unique()}
elev_norm = samples["status_elevador"].map(_map_elev).fillna("")
piloto_norm = samples["status_piloto"].map(_map_pil).fillna("")
valores_elev_ok = [remover_acentos(v) for v in VALOR_ELEVADOR_EFETIVO]
valores_on = [remover_acentos(v) for v in VALORES_LIGADO]

samples["elevador_forward"] = elev_norm.isin(valores_elev_ok)
samples["piloto_ligado"] = piloto_norm.isin(valores_on)
samples["motor_acima_limiar"] = samples["carga_motor"] >= LIMIAR_CARGA_MOTOR
# Tempo efetivo = carga >= 35% E elevador "forward"
samples["efetivo"] = samples["motor_acima_limiar"] & samples["elevador_forward"]

# Substitui 'df' pela tabela de amostras no restante do fluxo
df = samples

print(f"\nLinhas de telemetria (formato longo): consolidadas em amostras.")
print(f"Total de amostras (ceqid + timestamp): {len(df):,}".replace(",", "."))
print(f"  Amostras com piloto ligado          : {int(df['piloto_ligado'].sum()):,}".replace(",", "."))
print(f"  Amostras em tempo efetivo (>=35% + fwd): {int(df['efetivo'].sum()):,}".replace(",", "."))
df[["ceqid", "event_timestamp", "carga_motor", "status_elevador",
    "status_piloto", "efetivo", "piloto_ligado"]].head(10)


## 🌍 Etapa 4 — Conversão espacial (lat/lon → pontos geográficos)

Cria um `GeoDataFrame` de pontos usando **EPSG:4326 (WGS84)**. A construção da
geometria é feita de forma **vetorizada** com `gpd.points_from_xy`, adequada a
grandes volumes. A partir daqui trabalhamos com a **tabela de amostras** (uma
linha por posição/instante) construída na Etapa 3.5.


In [ ]:
# Criacao vetorizada da geometria de pontos (rapido para centenas de milhares)
pontos_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs=f"EPSG:{EPSG_GEO}",
)
print(f"GeoDataFrame de pontos criado: {len(pontos_gdf):,} pontos.".replace(",", "."))
print("CRS dos pontos:", pontos_gdf.crs)
pontos_gdf[["ceqid", "lat", "lon", "event_timestamp", "geometry"]].head()


## 🔗 Etapa 5 — Relacionamento espacial (Point-in-Polygon)

Executa o **Spatial Join** do GeoPandas (`sjoin`, predicado `within`), que usa
**índice espacial (R-tree)** automaticamente. Para cada ponto são anexados o
**nome**, o **ID** e **todos os atributos** da área correspondente — sem loops
linha a linha.

Pontos que caem em mais de um polígono (áreas sobrepostas) são resolvidos
mantendo a primeira correspondência para não duplicar registros.


In [ ]:
# Colunas da area que queremos trazer para cada ponto
cols_area = ["area_id", "area_nome"] + ATRIBUTOS_KML
areas_join = areas_gdf[cols_area + ["geometry"]].copy()

# Renomeia atributos do KML para evitar colisao com colunas dos pontos
renomear = {}
for c in ATRIBUTOS_KML:
    if c in pontos_gdf.columns:
        renomear[c] = f"kml_{c}"
areas_join = areas_join.rename(columns=renomear)
cols_area_final = ["area_id", "area_nome"] + [renomear.get(c, c) for c in ATRIBUTOS_KML]

# Spatial join vetorizado com indice espacial (predicate='within')
print("Executando Spatial Join (Point-in-Polygon) com indice espacial...")
juntado = gpd.sjoin(
    pontos_gdf, areas_join, how="left", predicate="within"
)

# Em caso de sobreposicao, um ponto pode duplicar; mantem a 1a area encontrada
juntado = juntado[~juntado.index.duplicated(keep="first")].copy()
juntado = juntado.drop(columns=[c for c in ["index_right"] if c in juntado.columns])

# Marca pontos sem area
juntado["area_nome"] = juntado["area_nome"].fillna("SEM AREA")
sem_area = int((juntado["area_nome"] == "SEM AREA").sum())

print(f"Pontos relacionados a alguma area: {len(juntado) - sem_area:,}".replace(",", "."))
print(f"Pontos SEM area associada        : {sem_area:,}".replace(",", "."))
juntado[["ceqid", "event_timestamp", "area_nome", "area_id"]].head()


## ⏱️ Etapa 6 — Cálculo do tempo trabalhado

Ordena por `ceqid` e `event_timestamp` e calcula a diferença de tempo entre
pontos consecutivos **do mesmo equipamento** (vetorizado com `groupby().diff()`).

**Regras aplicadas:**
- Intervalos **negativos** são ignorados (timestamps fora de ordem).
- Intervalos **maiores que 10 minutos** são ignorados (equipamento parado/desligado).
- O tempo é atribuído ao ponto **atual** (e à sua área), representando o trabalho
  realizado desde o ponto anterior.

São criadas as colunas `tempo_segundos`, `tempo_minutos` e `tempo_horas`.

Além do tempo total, o mesmo intervalo é classificado por **estado de operação**
(definido na Etapa 3.5), gerando:

- `tempo_efetivo_segundos` / `tempo_efetivo_horas` — **tempo efetivo** (carga ≥ 35% e elevador `forward`).
- `tempo_piloto_segundos` / `tempo_piloto_horas` — **tempo com piloto ligado** (orientação automática `on`).


In [ ]:
# Ordena por equipamento e tempo (essencial para o diff temporal)
juntado = juntado.sort_values(["ceqid", "event_timestamp"]).reset_index(drop=True)

# Diferenca de tempo (em segundos) em relacao ao ponto anterior do MESMO ceqid
delta = juntado.groupby("ceqid")["event_timestamp"].diff().dt.total_seconds()

# Aplica as regras: descarta negativos e gaps > 10 min; NaN (1o ponto) -> 0
delta = delta.where((delta >= 0) & (delta <= MAX_GAP_SECONDS))
juntado["tempo_segundos"] = delta.fillna(0.0)
juntado["tempo_minutos"] = juntado["tempo_segundos"] / 60.0
juntado["tempo_horas"] = juntado["tempo_segundos"] / 3600.0

# Tempo por ESTADO de operacao: o intervalo so conta para o estado se a
# amostra atual estiver naquele estado (efetivo / piloto ligado).
juntado["tempo_efetivo_segundos"] = np.where(
    juntado["efetivo"], juntado["tempo_segundos"], 0.0
)
juntado["tempo_efetivo_horas"] = juntado["tempo_efetivo_segundos"] / 3600.0

juntado["tempo_piloto_segundos"] = np.where(
    juntado["piloto_ligado"], juntado["tempo_segundos"], 0.0
)
juntado["tempo_piloto_horas"] = juntado["tempo_piloto_segundos"] / 3600.0

total_horas = juntado["tempo_horas"].sum()
total_efetivo = juntado["tempo_efetivo_horas"].sum()
total_piloto = juntado["tempo_piloto_horas"].sum()
print(f"Tempo total valido    : {total_horas:,.2f} h".replace(",", "."))
print(f"Tempo EFETIVO (>=35% + elevador forward): {total_efetivo:,.2f} h".replace(",", "."))
print(f"Tempo PILOTO (orientacao automatica on) : {total_piloto:,.2f} h".replace(",", "."))
juntado[["ceqid", "event_timestamp", "area_nome", "carga_motor",
         "status_elevador", "status_piloto", "tempo_horas",
         "tempo_efetivo_horas", "tempo_piloto_horas"]].head(10)


## 📊 Etapa 7 — Resumo por área

Agrega por área: quantidade de pontos, horas trabalhadas, **horas efetivas**
(carga ≥ 35% + elevador `forward`), **horas com piloto ligado**, % de
aproveitamento efetivo, tempo total (segundos e horas), número de equipamentos
distintos e o primeiro/último registro.


In [ ]:
resumo_area = (
    juntado.groupby("area_nome", dropna=False)
    .agg(
        area_id=("area_id", "first"),
        qtd_pontos=("event_timestamp", "size"),
        tempo_total_segundos=("tempo_segundos", "sum"),
        horas_trabalhadas=("tempo_horas", "sum"),
        horas_efetivas=("tempo_efetivo_horas", "sum"),
        horas_piloto=("tempo_piloto_horas", "sum"),
        qtd_equipamentos=("ceqid", "nunique"),
        primeiro_registro=("event_timestamp", "min"),
        ultimo_registro=("event_timestamp", "max"),
    )
    .reset_index()
)

# Duplicamos horas em coluna explicita "tempo_total_horas" para clareza no relatorio
resumo_area["tempo_total_horas"] = resumo_area["horas_trabalhadas"]
# % de aproveitamento efetivo sobre o tempo total trabalhado
resumo_area["pct_efetivo"] = np.where(
    resumo_area["horas_trabalhadas"] > 0,
    100 * resumo_area["horas_efetivas"] / resumo_area["horas_trabalhadas"], 0.0
)

# Reordena as colunas conforme a entrega esperada
resumo_area = resumo_area[[
    "area_nome", "area_id", "qtd_pontos", "horas_trabalhadas",
    "horas_efetivas", "horas_piloto", "pct_efetivo",
    "tempo_total_segundos", "tempo_total_horas", "qtd_equipamentos",
    "primeiro_registro", "ultimo_registro",
]].sort_values("horas_trabalhadas", ascending=False).reset_index(drop=True)

# Arredonda para leitura
for c in ["horas_trabalhadas", "horas_efetivas", "horas_piloto",
          "tempo_total_horas", "pct_efetivo"]:
    resumo_area[c] = resumo_area[c].round(2)
resumo_area["tempo_total_segundos"] = resumo_area["tempo_total_segundos"].round(0)

print(f"Resumo gerado para {len(resumo_area)} area(s).")
resumo_area


## 🚜 Etapa 8 — Resumo por equipamento

Agrega por **equipamento e área**, trazendo `ceqid`, `nickname`, `vin`, área,
horas trabalhadas, **horas efetivas**, **horas com piloto ligado** e quantidade
de pontos.


In [ ]:
def primeiro_nao_vazio(serie):
    """Retorna o primeiro valor nao nulo/nao vazio de uma serie (para nickname/vin)."""
    s = serie.dropna().astype(str)
    s = s[s.str.strip() != ""]
    return s.iloc[0] if len(s) else ""

resumo_equip = (
    juntado.groupby(["ceqid", "area_nome"], dropna=False)
    .agg(
        nickname=("nickname", primeiro_nao_vazio),
        vin=("vin", primeiro_nao_vazio),
        horas_trabalhadas=("tempo_horas", "sum"),
        horas_efetivas=("tempo_efetivo_horas", "sum"),
        horas_piloto=("tempo_piloto_horas", "sum"),
        qtd_pontos=("event_timestamp", "size"),
    )
    .reset_index()
)

# Reordena conforme a entrega esperada
resumo_equip = resumo_equip[[
    "ceqid", "nickname", "vin", "area_nome", "horas_trabalhadas",
    "horas_efetivas", "horas_piloto", "qtd_pontos",
]].sort_values(
    ["ceqid", "horas_trabalhadas"], ascending=[True, False]
).reset_index(drop=True)

for c in ["horas_trabalhadas", "horas_efetivas", "horas_piloto"]:
    resumo_equip[c] = resumo_equip[c].round(2)

print(f"Resumo gerado para {resumo_equip['ceqid'].nunique()} equipamento(s).")
resumo_equip.head(20)


## 💾 Etapa 9 — Relatório final + download

Exporta os três relatórios e os disponibiliza para download.

> ⚠️ **Atenção ao limite do Excel:** uma planilha `.xlsx` aceita no máximo
> **1.048.576 linhas**. Como a base de pontos pode ter **milhões** de registros,
> a tabela de pontos é exportada automaticamente em **CSV** (`resultado_pontos.csv`)
> quando ultrapassa esse limite — o CSV não tem limite de linhas e abre em
> Excel, Power BI, pandas etc. Se couber, é exportada em `.xlsx`. Os dois
> resumos (área e equipamento) são sempre pequenos e saem em `.xlsx`.


In [ ]:
# Limite maximo de linhas de uma aba do Excel (.xlsx)
LIMITE_LINHAS_EXCEL = 1_048_575  # 1.048.576 - 1 (cabecalho)

# Prepara a tabela de pontos para exportacao (sem a coluna geometry)
pontos_export = pd.DataFrame(juntado.drop(columns=["geometry"]))

def _df_excel_safe(df_in):
    """Converte colunas datetime com timezone para texto (Excel/CSV-friendly)."""
    out = df_in.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            try:
                out[col] = out[col].dt.tz_convert("UTC").astype(str)
            except (TypeError, AttributeError):
                out[col] = out[col].astype(str)
    return out

ARQ_AREA = "resumo_por_area.xlsx"
ARQ_EQUIP = "resumo_por_equipamento.xlsx"

pontos_safe = _df_excel_safe(pontos_export)

# Decide o formato da tabela de pontos conforme o numero de linhas
arquivos_download = []
if len(pontos_safe) > LIMITE_LINHAS_EXCEL:
    ARQ_PONTOS = "resultado_pontos.csv"
    print(
        f"Tabela de pontos com {len(pontos_safe):,} linhas excede o limite do "
        f"Excel; exportando em CSV...".replace(",", ".")
    )
    pontos_safe.to_csv(ARQ_PONTOS, index=False, encoding="utf-8-sig")
else:
    ARQ_PONTOS = "resultado_pontos.xlsx"
    print("Gerando 'resultado_pontos.xlsx'...")
    pontos_safe.to_excel(ARQ_PONTOS, index=False, engine="openpyxl")
arquivos_download.append(ARQ_PONTOS)

# Resumos sempre cabem no Excel
print("Gerando resumos em Excel...")
_df_excel_safe(resumo_area).to_excel(ARQ_AREA, index=False, engine="openpyxl")
_df_excel_safe(resumo_equip).to_excel(ARQ_EQUIP, index=False, engine="openpyxl")
arquivos_download += [ARQ_AREA, ARQ_EQUIP]

print("\nArquivos gerados:")
for a in arquivos_download:
    print(f"  {a}  ({os.path.getsize(a)/1024/1024:.2f} MB)")

# Disponibiliza para download no Colab
if EM_COLAB:
    for a in arquivos_download:
        files.download(a)


## 🗺️ Etapa 10 — Visualização (mapa Folium)

Gera um mapa interativo com os **polígonos do KML** e os **pontos GPS**. Cada
ponto tem um popup com `ceqid`, `nickname`, `event_timestamp` e a área.

Para manter o mapa leve com grandes volumes, os pontos são **amostrados**
(parâmetro `MAX_PONTOS_MAPA`). Aumente o limite se quiser ver todos — porém o
arquivo HTML pode ficar pesado.


In [ ]:
MAX_PONTOS_MAPA = 5000   # amostragem para manter o mapa leve

# Centro do mapa: centroide medio das areas (ou media dos pontos)
try:
    centro = areas_gdf.geometry.union_all().centroid
    centro_latlon = [centro.y, centro.x]
except Exception:
    centro_latlon = [juntado["lat"].mean(), juntado["lon"].mean()]

mapa = folium.Map(location=centro_latlon, zoom_start=13, tiles="OpenStreetMap")

# --- Camada de poligonos do KML ---
camada_areas = folium.FeatureGroup(name="Areas (KML)")
folium.GeoJson(
    areas_gdf[["area_nome", "geometry"]].to_json(),
    style_function=lambda x: {
        "color": "#1f78b4", "weight": 2, "fillColor": "#a6cee3", "fillOpacity": 0.25,
    },
    tooltip=folium.GeoJsonTooltip(fields=["area_nome"], aliases=["Area:"]),
).add_to(camada_areas)
camada_areas.add_to(mapa)

# --- Camada de pontos (amostrada) ---
camada_pontos = folium.FeatureGroup(name="Pontos GPS")
amostra = juntado
if len(juntado) > MAX_PONTOS_MAPA:
    amostra = juntado.sample(MAX_PONTOS_MAPA, random_state=42)
    print(f"Amostrando {MAX_PONTOS_MAPA:,} de {len(juntado):,} pontos no mapa.".replace(",", "."))

for _, r in amostra.iterrows():
    popup = folium.Popup(
        html=(
            f"<b>ceqid:</b> {r['ceqid']}<br>"
            f"<b>nickname:</b> {r.get('nickname', '')}<br>"
            f"<b>event_timestamp:</b> {r['event_timestamp']}<br>"
            f"<b>area:</b> {r['area_nome']}"
        ),
        max_width=300,
    )
    cor = "#e31a1c" if r["area_nome"] == "SEM AREA" else "#33a02c"
    folium.CircleMarker(
        location=[r["lat"], r["lon"]], radius=2,
        color=cor, fill=True, fill_opacity=0.7, popup=popup,
    ).add_to(camada_pontos)
camada_pontos.add_to(mapa)

folium.LayerControl().add_to(mapa)

ARQ_MAPA = "mapa.html"
mapa.save(ARQ_MAPA)
print(f"Mapa salvo em '{ARQ_MAPA}' ({os.path.getsize(ARQ_MAPA)/1024:.1f} KB).")

if EM_COLAB:
    files.download(ARQ_MAPA)

# Exibe o mapa dentro do notebook
mapa


## 📈 Etapa 11 — Métricas finais

Resumo geral do processamento.


In [ ]:
total_pontos = len(juntado)
total_areas = juntado.loc[juntado["area_nome"] != "SEM AREA", "area_nome"].nunique()
total_equip = juntado["ceqid"].nunique()
total_horas = juntado["tempo_horas"].sum()
total_efetivo = juntado["tempo_efetivo_horas"].sum()
total_piloto = juntado["tempo_piloto_horas"].sum()
pct_efetivo = (100 * total_efetivo / total_horas) if total_horas > 0 else 0.0
pct_piloto = (100 * total_piloto / total_horas) if total_horas > 0 else 0.0
pontos_sem_area = int((juntado["area_nome"] == "SEM AREA").sum())

print("=" * 56)
print("            METRICAS FINAIS DO PROCESSAMENTO")
print("=" * 56)
print(f"Total de pontos processados        : {total_pontos:,}".replace(",", "."))
print(f"Total de areas encontradas         : {total_areas:,}".replace(",", "."))
print(f"Total de equipamentos              : {total_equip:,}".replace(",", "."))
print(f"Total de horas calculadas          : {total_horas:,.2f} h".replace(",", "."))
print(f"Horas EFETIVAS (>=35% + forward)   : {total_efetivo:,.2f} h ({pct_efetivo:.1f}%)".replace(",", "."))
print(f"Horas com PILOTO ligado            : {total_piloto:,.2f} h ({pct_piloto:.1f}%)".replace(",", "."))
print(f"Pontos SEM area associada          : {pontos_sem_area:,}".replace(",", "."))
print("=" * 56)

# Tabela-resumo das metricas
pd.DataFrame({
    "Metrica": [
        "Total de pontos processados",
        "Total de areas encontradas",
        "Total de equipamentos",
        "Total de horas calculadas",
        "Horas efetivas (>=35% + elevador forward)",
        "Horas com piloto ligado (orientacao automatica)",
        "Pontos sem area associada",
    ],
    "Valor": [
        total_pontos, total_areas, total_equip,
        round(total_horas, 2), round(total_efetivo, 2),
        round(total_piloto, 2), pontos_sem_area,
    ],
})


---

✅ **Processamento concluído!** Os arquivos `resultado_pontos.xlsx`/`.csv`,
`resumo_por_area.xlsx`, `resumo_por_equipamento.xlsx` e `mapa.html` foram gerados
e baixados automaticamente. Os relatórios agora trazem, além das horas
trabalhadas, as **horas efetivas** (carga ≥ 35% + elevador `forward`) e as
**horas com piloto ligado** (orientação automática `on`). Se algum download não
iniciar, encontre os arquivos no painel de **Arquivos** (ícone de pasta) à
esquerda no Colab.
